# ML Mathematics Playground: 01. Linear Algebra

Linear Algebra is the absolute foundation of Machine Learning and AI. Every dataset is a matrix, every model parameter is a vector weight, and every layer transformation is matrix multiplication.

In this notebook, we will explore:
1. **Vector Operations** (Geometric view of addition & scaling)
2. **Linear Transformations** (Deforming 2D space with matrices)
3. **Eigenvalues & Eigenvectors** (Directions that remain invariant under transformations)

Let's import our libraries and set up the interactive widgets.


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, interactive

# Set clean style
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (6, 6)


## Section 1: Vector Operations

A vector can be represented as coordinates $[x, y]$ or geometrically as an arrow pointing from the origin $(0,0)$ to $(x, y)$.
- **Addition ($ec{u} + ec{v}$)**: Placed 'tip-to-tail', representing combined movements.
- **Scaling ($cec{u}$)**: Changes the length (magnitude) and/or direction (sign) of a vector.

Use the sliders below to adjust the vectors $ec{u}$ and $ec{v}$ and see their sum $ec{u} + ec{v}$ dynamically.


In [ ]:
def plot_vector_addition(ux, uy, vx, vy):
    u = np.array([ux, uy])
    v = np.array([vx, vy])
    w = u + v
    
    fig, ax = plt.subplots()
    
    # Draw grid lines and origin axes
    ax.axhline(0, color='grey', lw=1)
    ax.axvline(0, color='grey', lw=1)
    
    # Plot vectors
    ax.quiver(0, 0, u[0], u[1], angles='xy', scale_units='xy', scale=1, color='#1f77b4', label=f'u = [{ux}, {uy}]')
    ax.quiver(u[0], u[1], v[0], v[1], angles='xy', scale_units='xy', scale=1, color='#ff7f0e', label=f'v = [{vx}, {vy}] (tip-to-tail)')
    ax.quiver(0, 0, w[0], w[1], angles='xy', scale_units='xy', scale=1, color='#2ca02c', label=f'u + v = [{w[0]}, {w[1]}]')
    
    # Set limits and grid
    limit = max(10, np.max(np.abs([u, v, w])) + 2)
    ax.set_xlim(-limit, limit)
    ax.set_ylim(-limit, limit)
    ax.set_aspect('equal')
    ax.grid(True, which='both', linestyle='--', alpha=0.5)
    ax.legend(loc='upper left')
    ax.set_title('Vector Addition & Scaling Playground')
    plt.show()

interact(plot_vector_addition,
         ux=widgets.IntSlider(value=3, min=-10, max=10, step=1, description='u_x'),
         uy=widgets.IntSlider(value=4, min=-10, max=10, step=1, description='u_y'),
         vx=widgets.IntSlider(value=4, min=-10, max=10, step=1, description='v_x'),
         vy=widgets.IntSlider(value=-2, min=-10, max=10, step=1, description='v_y'));


## Section 2: Linear Transformations

A matrix multiplication $Aec{x}$ represents a **linear transformation** of space. The columns of the matrix $A$ define where the basis vectors $\hat{i} = [1, 0]^T$ and $\hat{j} = [0, 1]^T$ land after the transformation.

Let's define a matrix:
$$A = \begin{pmatrix} a & b \\ c & d \end{pmatrix}$$

Use the sliders below to change $a, b, c, d$ and watch how the unit grid transforms. Notice how rotation, shearing, scaling, and reflection are achieved by choosing different values.


In [ ]:
def plot_linear_transformation(a, b, c, d):
    # Create a grid of points
    x = np.linspace(-5, 5, 11)
    y = np.linspace(-5, 5, 11)
    xx, yy = np.meshgrid(x, y)
    grid_points = np.vstack([xx.flatten(), yy.flatten()])
    
    # Define Transformation Matrix
    A = np.array([[a, b], [c, d]])
    
    # Apply transformation
    transformed_points = A @ grid_points
    
    # Separate x and y
    tx = transformed_points[0].reshape(xx.shape)
    ty = transformed_points[1].reshape(yy.shape)
    
    fig, ax = plt.subplots(figsize=(7, 7))
    
    # Plot original grid lines lightly
    for i in range(xx.shape[0]):
        ax.plot(xx[i, :], yy[i, :], color='lightblue', alpha=0.3, linestyle=':')
        ax.plot(xx[:, i], yy[:, i], color='lightblue', alpha=0.3, linestyle=':')
        
    # Plot transformed grid lines
    for i in range(tx.shape[0]):
        ax.plot(tx[i, :], ty[i, :], color='#1f77b4', alpha=0.7)
        ax.plot(tx[:, i], ty[:, i], color='#1f77b4', alpha=0.7)
        
    # Plot new basis vectors
    ax.quiver(0, 0, a, c, angles='xy', scale_units='xy', scale=1, color='red', width=0.015, label='Transformed i-hat')
    ax.quiver(0, 0, b, d, angles='xy', scale_units='xy', scale=1, color='green', width=0.015, label='Transformed j-hat')
    
    ax.axhline(0, color='black', lw=0.8)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_xlim(-10, 10)
    ax.set_ylim(-10, 10)
    ax.set_aspect('equal')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='upper left')
    ax.set_title(f'Transformation Matrix A = [[{a}, {b}], [{c}, {d}]]')
    plt.show()

interact(plot_linear_transformation,
         a=widgets.FloatSlider(value=1.0, min=-3.0, max=3.0, step=0.1, description='a (i_x)'),
         b=widgets.FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.1, description='b (j_x)'),
         c=widgets.FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.1, description='c (i_y)'),
         d=widgets.FloatSlider(value=1.0, min=-3.0, max=3.0, step=0.1, description='d (j_y)'));


## Section 3: Eigenvalues & Eigenvectors

For a matrix $A$, eigenvectors $ec{v}$ are vectors whose direction **does not change** when transformed by $A$. They are only scaled by a factor $\lambda$ (the eigenvalue):
$$Aec{v} = \lambda ec{v}$$

### Interactive Eigenvector Hunt!
Below is a matrix $A = \begin{pmatrix} 2 & 1 \\ 1 & 2 \end{pmatrix}$.
Use the slider to rotate an input vector $ec{x}$ (red). Watch the output vector $Aec{x}$ (blue).
Can you find the angles where the red and blue vectors line up exactly? Those are the eigenvectors!


In [ ]:
# Fixed matrix
A = np.array([[2.0, 1.0], [1.0, 2.0]])

# Find actual eigenvectors for reference line display
evals, evecs = np.linalg.eig(A)

def hunt_eigenvectors(theta_deg):
    theta = np.radians(theta_deg)
    # Input vector
    x = np.array([np.cos(theta), np.sin(theta)]) * 3
    # Output vector
    Ax = A @ x
    
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.axhline(0, color='grey', lw=1)
    ax.axvline(0, color='grey', lw=1)
    
    # Plot input vector
    ax.quiver(0, 0, x[0], x[1], angles='xy', scale_units='xy', scale=1, color='red', width=0.012, label='Input Vector x')
    # Plot output vector
    ax.quiver(0, 0, Ax[0], Ax[1], angles='xy', scale_units='xy', scale=1, color='blue', width=0.012, label='Output Vector Ax')
    
    # Check if they are collinear (dot product test)
    unit_x = x / np.linalg.norm(x)
    unit_Ax = Ax / np.linalg.norm(Ax)
    collinear = np.abs(np.dot(unit_x, unit_Ax)) > 0.999
    
    if collinear:
        lambda_approx = np.linalg.norm(Ax) / np.linalg.norm(x)
        # check direction
        if np.dot(unit_x, unit_Ax) < 0:
            lambda_approx = -lambda_approx
        ax.set_title(f'🎉 EIGENVECTOR FOUND! lambda ≈ {lambda_approx:.2f}', color='green', fontsize=14, fontweight='bold')
    else:
        ax.set_title('Rotate x to align with Ax')
        
    ax.set_xlim(-10, 10)
    ax.set_ylim(-10, 10)
    ax.set_aspect('equal')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='upper left')
    plt.show()

interact(hunt_eigenvectors, theta_deg=widgets.IntSlider(value=0, min=0, max=360, step=5, description='Angle (deg)'));
